In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Import Libraries

In [ ]:
pip install sentence-transformers transformers datasets accelerate

In [ ]:
import pandas as pd
import numpy as np
import string

from transformers import AutoTokenizer, AutoModelForSequenceClassification,TrainingArguments, Trainer
from datasets import Dataset


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# EDA

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

In [ ]:
train.head()

In [ ]:
train['answer'].value_counts()

In [ ]:
train.isnull().sum()

# Evaluation Metric

In [ ]:
def map3(actuals, predictions):
    score = 0

    for actual, pred in zip(actuals, predictions):
        if actual == pred[0]:
            score += 1
        elif actual == pred[1]:
            score += 1/2
        elif actual == pred[2]:
            score += 1/3

    return score / len(actuals)

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Model 2 (Pretrained): DeBERTa

In [ ]:
train_questions, val_questions = train_test_split(
    train,
    test_size=0.2,
    stratify=train["answer"],
    random_state=4524
)

print(train_questions.shape)
print(val_questions.shape)

In [ ]:
def expand_mcq(df):

    rows = []

    for _, row in df.iterrows():

        correct = row["answer"]

        for choice in ["A", "B", "C", "D", "E"]:

            rows.append({
                "prompt": row["prompt"],
                "option": row[choice],
                "label": int(choice == correct)
            })

    return pd.DataFrame(rows)

In [ ]:
train_bert = expand_mcq(train_questions)
val_bert = expand_mcq(val_questions)

print(train_bert.shape)
print(val_bert.shape)

In [ ]:
model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
train_ds = Dataset.from_pandas(train_bert)
val_ds = Dataset.from_pandas(val_bert)

In [ ]:
def tokenize(batch):

    return tokenizer(
        batch["prompt"],
        batch["option"],
        truncation=True,
        padding="max_length",
        max_length=384
    )

In [ ]:
train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

In [ ]:
train_ds = train_ds.remove_columns(
    ["prompt", "option"]
)

val_ds = val_ds.remove_columns(
    ["prompt", "option"]
)

train_ds = train_ds.rename_column(
    "label",
    "labels"
)

val_ds = val_ds.rename_column(
    "label",
    "labels"
)

In [ ]:
train_ds.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

val_ds.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    torch_dtype=torch.float32
)

model = model.cuda()

In [ ]:
training_args = TrainingArguments(
    output_dir="./deberta_base",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

In [ ]:
trainer.train()

In [ ]:
choices = ["A", "B", "C", "D", "E"]

def predict_question(row):

    encodings = tokenizer(
        [row["prompt"]] * 5,
        [row[c] for c in choices],
        padding=True,
        truncation=True,
        max_length=384,
        return_tensors="pt"
    )

    encodings = {
        k: v.to(model.device)
        for k, v in encodings.items()
    }

    with torch.no_grad():

        outputs = model(**encodings)

        probs = torch.softmax(
            outputs.logits,
            dim=1
        )[:, 1]

    probs = probs.cpu().numpy()

    order = np.argsort(probs)[::-1]

    return [choices[i] for i in order[:3]]

In [ ]:
val_predictions = []

for _, row in val_questions.iterrows():

    val_predictions.append(
        predict_question(row)
    )

score = map3(
    val_questions["answer"].tolist(),
    val_predictions
)

print("Validation MAP@3:", score)

In [ ]:
test_predictions = []

for _, row in test.iterrows():

    test_predictions.append(
        " ".join(
            predict_question(row)
        )
    )

# Submission Cell

In [ ]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()